# Tutorial 03 -- Uncertainty Quantification

Monte Carlo propagation, Hermite polynomial chaos, and grid Bayesian calibration applied to a PISA monopile.

## MC propagation

In [ ]:
from op3.uq import SoilPrior, propagate_pisa_mc, summarise_samples

priors = [
    SoilPrior(0.0,  5.0e7, G_cov=0.30, soil_type='sand'),
    SoilPrior(15.0, 1.0e8, G_cov=0.30, soil_type='sand'),
    SoilPrior(36.0, 1.5e8, G_cov=0.30, soil_type='sand'),
]
samples = propagate_pisa_mc(diameter_m=6.0, embed_length_m=36.0, soil_priors=priors, n_samples=500)
print(f'Sample shape: {samples.shape}')
summary = summarise_samples(samples)
for k in ('Kxx', 'Krxrx'):
    s = summary[k]
    print(f"{k}: mean={s['mean']:.2e}, COV={s['cov']:.2f}, 5%={s['p05']:.2e}, 95%={s['p95']:.2e}")

## Hermite PCE surrogate

In [ ]:
import numpy as np
from op3.uq import build_pce_1d, pce_mean_var

# Surrogate of a nonlinear response: f(xi) = exp(0.3 * xi) where xi ~ N(0,1)
pce = build_pce_1d(lambda xi: np.exp(0.3 * xi), order=6)
mean, var = pce_mean_var(pce)
print(f'PCE mean = {mean:.6f} (analytic = {np.exp(0.045):.6f})')
print(f'PCE std  = {np.sqrt(var):.6f}')

# Evaluate at a new point in microseconds, no solver call needed
print(f'Surrogate at xi=0.7: {pce.evaluate(0.7):.4f}')
print(f'True at xi=0.7:      {np.exp(0.21):.4f}')

## Bayesian calibration of tower EI

In [ ]:
from op3 import build_foundation, compose_tower_model
from op3.uq import grid_bayesian_calibration, normal_likelihood
from op3.opensees_foundations import tower_loader as TL
from dataclasses import replace

def forward(scale):
    orig = TL.load_elastodyn_tower
    def patched(*a, **kw):
        t = orig(*a, **kw)
        return replace(t, ei_fa_Nm2=t.ei_fa_Nm2*scale, ei_ss_Nm2=t.ei_ss_Nm2*scale)
    TL.load_elastodyn_tower = patched
    try:
        m = compose_tower_model(rotor='nrel_5mw_baseline', tower='nrel_5mw_oc3_tower',
                                 foundation=build_foundation(mode='fixed'))
        return float(m.eigen(n_modes=3)[0])
    finally:
        TL.load_elastodyn_tower = orig

post = grid_bayesian_calibration(
    forward_model=forward,
    likelihood_fn=normal_likelihood(measured=0.2766, sigma=0.01),
    grid=np.linspace(0.7, 1.3, 31),
)
print(f'EI scale posterior: mean={post.mean:.3f} +/- {post.std:.3f}')
print(f'5%-95%: [{post.p05:.3f}, {post.p95:.3f}]')